# Forest Segmentation — U-Net Baseline (Colab)

Week 1-2 goal: train a plain U-Net from scratch (no pretrained encoder, no
attention, no augmentation) on the full dataset and report Dice / IoU.

This notebook is a deliberate structural clone of
`Notebooks/Kalana/forest_segformer_colab.ipynb`: the dataset class, the
train/val/test split (same seed, same file lists), the loss, the metric and the
training loop are all identical. **Only the model is swapped** — SegFormer-B0
out, vanilla U-Net in. That makes the final test Dice / IoU directly comparable
to Kalana's SegFormer number, which is what `plan.txt` Step 1 asks for.

Steps 7 and 8 at the end add the training curves, prediction grid and
Params / GFLOPs figures needed for the Phase 6 ablation table.

**Before running:** make sure Runtime → Change runtime type → GPU (T4) is selected.

## Step 0: Mount Google Drive

Upload your `Chanupa` folder (with `images/` and `masks/` subfolders) to Google
Drive first, then mount it here. Adjust `DATA_ROOT` below to match wherever
you placed it in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q thop

## Step 1: Config + verify filename pairing

In [ ]:
import os
from pathlib import Path

# ── Config ────────────────────────────────────────────────
# Adjust this path to wherever "Chanupa" ended up inside your Drive.
# Example: "/content/drive/MyDrive/Chanupa"
DATA_ROOT = Path("/content/drive/MyDrive/Chanupa")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3  # U-Net trains from scratch; Kalana's 6e-5 is a fine-tuning LR
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
CHECKPOINT_OUT = "/content/drive/MyDrive/unet_baseline.pt"
RESULTS_DIR = Path("/content/drive/MyDrive/unet_baseline_results")
# ──────────────────────────────────────────────────────────

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

image_files = sorted(os.listdir(IMAGES_DIR))
mask_files = sorted(os.listdir(MASKS_DIR))

print(f"Found {len(image_files)} images")
print(f"Found {len(mask_files)} masks")

In [ ]:
def mask_name_to_image_name(mask_filename):
    """Converts '855_mask_01.jpg' -> '855_sat_01.jpg'."""
    return mask_filename.replace("_mask", "_sat")

# Full-dataset pairing check before we trust it on everything
missing = []
for m in mask_files:
    expected_image = mask_name_to_image_name(m)
    if expected_image not in image_files:
        missing.append((m, expected_image))

print(f"Total masks: {len(mask_files)}")
print(f"Missing matches: {len(missing)}")
if missing:
    print("First few missing pairs:", missing[:5])
assert len(missing) == 0, "Fix missing pairs before continuing."

## Step 2: Dataset class

Identical to Kalana's — resize, normalize, binarize the mask. Do not change
anything here: any drift breaks comparability with the SegFormer run.

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

class ForestSegDataset(Dataset):
    def __init__(self, mask_filenames, images_dir, masks_dir, img_size=256):
        self.mask_filenames = mask_filenames
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size

        # ImageNet normalization stats (kept identical to the SegFormer baseline
        # so both models see exactly the same inputs)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_fname = self.mask_filenames[idx]
        image_fname = mask_name_to_image_name(mask_fname)

        image = Image.open(self.images_dir / image_fname).convert("RGB")
        image = image.resize((self.img_size, self.img_size))

        mask = Image.open(self.masks_dir / mask_fname).convert("L")
        mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        # White (>127) = forest = 1, black = non-forest = 0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()

        return image, mask

## Step 3: Train / val / test split

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_splits():
    all_files = sorted(mask_files)
    random.shuffle(all_files)

    n = len(all_files)
    n_val = int(n * VAL_SPLIT)
    n_test = int(n * TEST_SPLIT)

    val_files = all_files[:n_val]
    test_files = all_files[n_val:n_val + n_test]
    train_files = all_files[n_val + n_test:]
    return train_files, val_files, test_files

set_seed(SEED)
train_files, val_files, test_files = make_splits()
print(f"Train/Val/Test sizes: {len(train_files)}/{len(val_files)}/{len(test_files)}")

## Step 4: Model + Dice/IoU metric

This is the only cell that differs structurally from the SegFormer notebook.
Vanilla U-Net: 4 encoder stages (64/128/256/512), a 1024-channel bottleneck,
transposed-conv decoder with skip connections, and a 1x1 head.

The head emits **2 channels, not 1** — that keeps `F.cross_entropy` and
`argmax` identical to the SegFormer baseline instead of switching to
sigmoid + BCE.

`dice_iou_score` is copied verbatim from Kalana's notebook.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

class DoubleConv(nn.Module):
    """(Conv3x3 -> BN -> ReLU) x 2 — the standard U-Net block."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2, features=(64, 128, 256, 512)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder
        c = in_channels
        for f in features:
            self.downs.append(DoubleConv(c, f))
            c = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder: upsample, concat skip, then DoubleConv
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat((skip, x), dim=1)
            x = self.ups[i + 1](x)

        return self.final_conv(x)

def build_model():
    model = UNet(in_channels=3, out_channels=2, features=(64, 128, 256, 512))
    return model.to(device)

def dice_iou_score(pred_mask, true_mask, eps=1e-6):
    """pred_mask, true_mask: bool tensors, same shape. Dice/IoU for the
    'forest' (positive) class."""
    intersection = (pred_mask & true_mask).sum().float()
    pred_sum = pred_mask.sum().float()
    true_sum = true_mask.sum().float()
    union = pred_sum + true_sum - intersection

    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return dice.item(), iou.item()

## Step 5: DataLoaders + training loop

Same loop as the SegFormer baseline — plain AdamW, no scheduler, no AMP, no
gradient clipping — scaled to the full dataset for 20 epochs on GPU.

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

train_ds = ForestSegDataset(train_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
val_ds = ForestSegDataset(val_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
test_ds = ForestSegDataset(test_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(model.__class__.__name__, "ready")

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_dice, total_iou, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        for images, masks in tqdm(loader, leave=False):
            images, masks = images.to(device), masks.to(device)

            logits = model(images)
            # No-op for U-Net (already full resolution), kept so this loop stays
            # line-for-line comparable with the SegFormer one.
            logits = F.interpolate(
                logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )

            loss = F.cross_entropy(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1).bool()
            dice, iou = dice_iou_score(preds, masks.bool())

            total_loss += loss.item()
            total_dice += dice
            total_iou += iou
            n_batches += 1

    return total_loss / n_batches, total_dice / n_batches, total_iou / n_batches

In [ ]:
import csv

LOG_PATH = RESULTS_DIR / "training_log.csv"
FIELDS = ["epoch", "train_loss", "train_dice", "train_iou",
          "val_loss", "val_dice", "val_iou"]

history = []
best_val_dice = 0.0

with open(LOG_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_dice, train_iou = run_epoch(model, train_loader, optimizer)
        val_loss, val_dice, val_iou = run_epoch(model, val_loader)

        row = {
            "epoch": epoch,
            "train_loss": train_loss, "train_dice": train_dice, "train_iou": train_iou,
            "val_loss": val_loss, "val_dice": val_dice, "val_iou": val_iou,
        }
        history.append(row)
        writer.writerow(row)
        f.flush()  # survive a Colab disconnect

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} dice={train_dice:.4f} iou={train_iou:.4f} | "
            f"val_loss={val_loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f}"
        )

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), CHECKPOINT_OUT)
            print(f"  -> saved new best checkpoint ({CHECKPOINT_OUT})")

print(f"\nBest val dice: {best_val_dice:.4f}")
print(f"Log written to: {LOG_PATH}")

## Step 6: Final test-set evaluation

This Dice/IoU number is your Week 1-2 deliverable to report to the team.
**Do not skip this cell** — the SegFormer run never executed its equivalent and
ended up with no test number.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_OUT, map_location=device))
test_loss, test_dice, test_iou = run_epoch(model, test_loader)
print(f"\nFINAL TEST RESULTS: dice={test_dice:.4f}  iou={test_iou:.4f}")

METRICS_PATH = RESULTS_DIR / "test_metrics.txt"
with open(METRICS_PATH, "w") as f:
    f.write("U-Net baseline (from scratch) - Forest Segmented, full dataset\n")
    f.write(f"train/val/test = {len(train_files)}/{len(val_files)}/{len(test_files)}\n")
    f.write(f"epochs={EPOCHS} batch_size={BATCH_SIZE} lr={LR} img_size={IMG_SIZE} seed={SEED}\n")
    f.write(f"best_val_dice: {best_val_dice:.4f}\n")
    f.write(f"test_loss: {test_loss:.4f}\n")
    f.write(f"test_dice: {test_dice:.4f}\n")
    f.write(f"test_iou:  {test_iou:.4f}\n")

print(f"Saved to: {METRICS_PATH}")

## Step 7: Training curves + qualitative predictions

In [ ]:
import matplotlib.pyplot as plt

epochs = [r["epoch"] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, [r["train_loss"] for r in history], label="train")
axes[0].plot(epochs, [r["val_loss"] for r in history], label="val")
axes[0].set_title("Cross-entropy loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [r["train_dice"] for r in history], label="train dice")
axes[1].plot(epochs, [r["val_dice"] for r in history], label="val dice")
axes[1].plot(epochs, [r["train_iou"] for r in history], "--", label="train iou")
axes[1].plot(epochs, [r["val_iou"] for r in history], "--", label="val iou")
axes[1].set_title("Dice / IoU (forest class)")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle("U-Net baseline — training curves")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
plt.show()

In [ ]:
N_SHOW = 4
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

model.eval()
fig, axes = plt.subplots(N_SHOW, 3, figsize=(9, 3 * N_SHOW))

with torch.no_grad():
    for row in range(N_SHOW):
        image, mask = test_ds[row]
        pred = model(image.unsqueeze(0).to(device)).argmax(dim=1)[0].cpu().numpy()

        rgb = image.permute(1, 2, 0).numpy() * std + mean
        rgb = np.clip(rgb, 0, 1)

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_ylabel(test_files[row], fontsize=7)
        axes[row, 1].imshow(mask.numpy(), cmap="gray")
        axes[row, 2].imshow(pred, cmap="gray")

        for col in range(3):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])

axes[0, 0].set_title("input")
axes[0, 1].set_title("ground truth")
axes[0, 2].set_title("prediction")

fig.suptitle("U-Net baseline — test-set predictions")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "prediction_grid.png", dpi=150)
plt.show()

## Step 8: Params + GFLOPs (for the ablation table)

`plan.txt` Phase 6 wants an ablation table with mIoU / Params / GFLOPs columns.
These are the two complexity numbers for this baseline row.

In [ ]:
from thop import profile

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
macs, _ = profile(build_model(), inputs=(dummy,), verbose=False)
gflops = macs * 2 / 1e9

summary = (
    f"params_total: {total_params/1e6:.2f} M\n"
    f"params_trainable: {trainable_params/1e6:.2f} M\n"
    f"gflops @ 1x3x{IMG_SIZE}x{IMG_SIZE}: {gflops:.2f}\n"
)
print(summary)

with open(RESULTS_DIR / "test_metrics.txt", "a") as f:
    f.write(summary)